# Quick Simulation Test Setup

Run the first code cell once. After that, use the next cells to inspect outputs, test queries, or debug modules.


In [1]:
# Quick Simulation Test Setup
#
# Run this cell once. After it finishes, you can test against:
# sim, orders_df, lines_df, po_df, daily_df, forecast_df, analytics_tables, database_result

from pathlib import Path
import sys
import sqlite3
import pandas as pd

# ---------------------------------------------------------
# Project path setup
# ---------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation
from simulation.forecasting import generate_baseline_forecast, calculate_forecast_metrics
from simulation.analytics import export_analytics_tables
from simulation.database_builder import build_sqlite_database

# ---------------------------------------------------------
# Simulation settings
# ---------------------------------------------------------

START_DATE = "2021-01-01"
END_DATE = "2025-12-31"
LOOKBACK_MONTHS = 3
REBUILD_SQLITE_DATABASE = True

# ---------------------------------------------------------
# Run simulation
# ---------------------------------------------------------

sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date=START_DATE,
    end_date=END_DATE,
)

forecast_df = generate_baseline_forecast(
    sim,
    lookback_months=LOOKBACK_MONTHS,
)

forecast_metrics = calculate_forecast_metrics(forecast_df)

analytics_tables = export_analytics_tables(sim)

sim.export_tables()

if REBUILD_SQLITE_DATABASE:
    database_result = build_sqlite_database()
else:
    database_result = None

# ---------------------------------------------------------
# Convenience DataFrames for testing
# ---------------------------------------------------------

orders_df = pd.DataFrame(sim.sales_orders)
lines_df = pd.DataFrame(sim.sales_order_lines)
po_df = pd.DataFrame(sim.purchase_orders)
daily_df = pd.DataFrame(sim.daily_order_summary)
inventory_history_df = pd.DataFrame(sim.inventory_history)

# ---------------------------------------------------------
# Quick status summary
# ---------------------------------------------------------

print("\nSimulation complete.")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Sales orders: {len(orders_df):,}")
print(f"Sales order lines: {len(lines_df):,}")
print(f"Purchase orders: {len(po_df):,}")
print(f"Inventory history rows: {len(inventory_history_df):,}")
print(f"Forecast rows: {len(forecast_df):,}")

print("\nDemand / fulfillment:")
print(f"Requested units: {lines_df['requested_qty'].sum():,}")
print(f"Fulfilled units: {lines_df['fulfilled_qty'].sum():,}")
print(f"Backordered units: {lines_df['backordered_qty'].sum():,}")

print("\nForecast metrics:")
print(forecast_metrics)

if database_result is not None:
    print("\nSQLite database:")
    print(database_result["database_path"])

print("\nReady for testing.")


Project root: /Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics

Simulation complete.
Date range: 2021-01-01 to 2025-12-31
Sales orders: 36,991
Sales order lines: 102,844
Purchase orders: 2,157
Inventory history rows: 306,810
Forecast rows: 9,450

Demand / fulfillment:
Requested units: 207,231
Fulfilled units: 129,974
Backordered units: 77,257

Forecast metrics:
{'forecast_rows': 9450, 'total_actual_qty': 197596, 'total_forecast_qty': 197401, 'total_absolute_error': 58459, 'wape': 0.2959, 'bias_pct': 0.001, 'mean_absolute_percentage_error': 0.5675}

SQLite database:
/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/database/sonoran_cycles.db

Ready for testing.


## Testing Area

Use the cells below for quick checks. The setup cell creates `sim`, `orders_df`, `lines_df`, `po_df`, `daily_df`, `forecast_df`, `analytics_tables`, and `database_result`.


In [2]:
from simulation.reporting import export_executive_summary

summary_path, summary_metrics = export_executive_summary(
    sim,
    analytics_tables,
)

summary_path

PosixPath('/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/reports/executive_insights.md')

In [11]:
# Example testing cell
#
# Start testing here. Replace these examples with whatever you want to inspect.

query = """
SELECT
    supplier_id,
    supplier_name,
    purchase_orders,
    open_purchase_orders,
    received_purchase_orders,
    ordered_units,
    received_units,
    open_units,
    ROUND(average_lead_time_days, 1) AS average_lead_time_days,
    ROUND(receipt_rate, 3) AS receipt_rate
FROM supplier_performance_summary
ORDER BY
    open_units DESC,
    ordered_units DESC;
"""

supplier_exposure = run_query(query)

supplier_exposure

,supplier_id,supplier_name,purchase_orders,open_purchase_orders,received_purchase_orders,ordered_units,received_units,open_units,average_lead_time_days,receipt_rate
0,S001,Merida Industry,2029,68,1961,124704,120531,4173,55.0,0.967
1,S002,Ideal Bike Corp,126,2,124,7684,7561,123,45.0,0.984


## Optional SQL Testing

Use this after the setup cell rebuilds the SQLite database.


In [7]:
# Optional SQL testing cell

query = """
SELECT
    supplier_id,
    supplier_name,
    purchase_orders,
    open_purchase_orders,
    received_purchase_orders,
    ordered_units,
    received_units,
    open_units,
    ROUND(average_lead_time_days, 1) AS average_lead_time_days,
    ROUND(receipt_rate, 3) AS receipt_rate
FROM supplier_performance_summary
ORDER BY
    open_units DESC,
    ordered_units DESC;
"""
